In [1]:
import cv2
import pyzed.sl as sl
import math
import numpy as np

In [ ]:
# Global variable 
# camera_settings = sl.VIDEO_SETTINGS.BRIGHTNESS
# str_camera_settings = "BRIGHTNESS" 
# step_camera_settings = 1
# led_on = True 
# selection_rect = sl.Rect()
# select_in_progress = False
# origin_rect = (-1,-1 )
# selected_points = []

In [ ]:
def on_mouse(event,x,y,selected_points):
    global select_in_progress,selection_rect,origin_rect
    global selected_point

    if event == cv2.EVENT_LBUTTONDOWN:
        selected_points.append((x,y))
    elif event == cv2.EVENT_RBUTTONDOWN and len(selected_points)>0:
        selected_points.pop()
    return selected_points


def print_camera_information(cam):
    cam_info = cam.get_camera_information()
    print("ZED Model                 : {0}".format(cam_info.camera_model))
    print("ZED Serial Number         : {0}".format(cam_info.serial_number))
    print("ZED Camera Firmware       : {0}/{1}".format(cam_info.camera_configuration.firmware_version,cam_info.sensors_configuration.firmware_version))
    print("ZED Camera Resolution     : {0}x{1}".format(round(cam_info.camera_configuration.resolution.width, 2), cam.get_camera_information().camera_configuration.resolution.height))
    print("ZED Camera FPS            : {0}".format(int(cam_info.camera_configuration.fps)))



In [ ]:
init = sl.InitParameters(depth_mode=sl.DEPTH_MODE.NEURAL, # NONE, PERFORMANCE, QUALITY, ULTRA, NEURAL, NEURAL_PLUS (HORRIBLY SLOW)
                                 coordinate_units=sl.UNIT.CENTIMETER,
                                 camera_resolution = sl.RESOLUTION.HD1200, #HD1200, HD1080, SVGA
                                 camera_fps = 60, #60,30,15, 120(SVGA only)
                                 depth_stabilization = 50 #reduce depth map jitter [0-100] (100 is too much, >=75 is fine. Produces some latent effect around the edges of the visual field)
                                 )
cam = sl.Camera()
status = cam.open(init)
if status != sl.ERROR_CODE.SUCCESS:
    print("Camera Open : "+repr(status)+". Exit program.")
    exit()


selected_points = []
runtime = sl.RuntimeParameters()
svo_image = sl.Mat() 
svo_depth_map = sl.Mat()
svo_confidence_map = sl.Mat()


win_name = "Camera Control"
cv2.namedWindow(win_name)
cv2.setMouseCallback(win_name,on_mouse)

key = ''
while key != 113:  # for 'q' key
    err = cam.grab(runtime) 

    if err == sl.ERROR_CODE.SUCCESS: # Check that a new image is successfully acquired

        cam.retrieve_image(svo_image, sl.VIEW.LEFT) # Retrieve left image
        cam.retrieve_measure(svo_depth_map, sl.MEASURE.DEPTH,sl.MEM.CPU, sl.Resolution(0,0))
        cam.retrieve_measure(svo_confidence_map, sl.MEASURE.CONFIDENCE)

        depth_map = np.transpose(svo_depth_map.get_data())
        confidence_map = np.transpose(svo_confidence_map.get_data())
        
        cvImage = cv2.flip(svo_image.get_data(),1) # Convert sl.Mat to cv2.Mat


        if len(selected_points) > 0:
            for point in selected_points:
                depth = depth_map[point]
                confidence = confidence_map[point]
                if not math.isnan(depth) and not math.isinf(depth) and not math.isnan(confidence):
                    depth = round(depth)
                    confidence = round(confidence)

                    cvImage = cv2.circle(cvImage,point,1,(255,255,255),1)
                    cvImage = cv2.putText(cvImage, f'{depth,confidence}', point, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)

        # cv2.imshow(win_name, cv2.resize(cvImage, (1440,900))) #Display image
        # cv2.imshow(win_name, cvImage) #Display image
        cv2.imshow(win_name, cv2.resize(svo_confidence_map.get_data(),(1440,900)))
    else:
        print("Error during capture : ", err)
        break
    
    key = cv2.waitKey(0)
    # Change camera settings with keyboard
    # update_camera_settings(key, cam, runtime, mat)
cv2.destroyAllWindows()

cam.close()

[2025-01-31 13:54:11 UTC][ZED][INFO] Logging level INFO
[2025-01-31 13:54:11 UTC][ZED][INFO] Logging level INFO
[2025-01-31 13:54:12 UTC][ZED][INFO] Logging level INFO
[2025-01-31 13:54:13 UTC][ZED][INFO] [Init]  Depth mode: NEURAL
[2025-01-31 13:54:15 UTC][ZED][INFO] [Init]  Camera FW version: 2001
[2025-01-31 13:54:15 UTC][ZED][INFO] [Init]  Video mode: HD1200@60
[2025-01-31 13:54:15 UTC][ZED][INFO] [Init]  Serial Number: S/N 43957242


In [5]:
(depth_map.shape, depth_map.shape, cvImage.shape)

((1920, 1200), (1920, 1200), (1200, 1920, 4))

In [6]:
np.array(depth_map.shape) * 0.75

array([1440.,  900.])

In [ ]:
math.isnum(depth)

In [ ]:
math.isinf(depth)